In [ ]:
from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.item_representation.content_embeddings import (
    AmazonItemTextBuilder,
    ContentEmbeddingBuilder,
    HashingTextEmbedder,
)
from pathlib import Path

from flowrec.item_representation.content_embeddings import ItemEmbeddingStore
from flowrec.item_representation.semantic_id.rqvae import (
    RQVAEConfig,
    RQVAETrainConfig)
from flowrec.item_representation.semantic_id.rqvae_tokenizer import  RQVAEItemTokenizer, RQVAEItemTokenizerConfig

In [ ]:
cfg = AmazonReviewDataConfig(
    reviews_path="../data/raw/2018/Musical_Instruments.json.gz",
    metadata_path="../data/raw/2018/meta_Musical_Instruments.json.gz",
    item_key="asin",

    min_user_interactions=5,
    min_item_interactions=5,

    split_strategy="global_time",
    global_val_ratio=0.1,
    global_test_ratio=0.1,
    global_history_policy="rolling",
    global_eval_target_mode="last_per_user",
    deduplicate_consecutive = True,

    max_history_len=20,
    batch_size=256,
    eval_batch_size=512,
)

dm = AmazonReviewDataModule(cfg)
dm.setup()

print(dm.summary())

In [ ]:
dm.item2id

In [ ]:
dm.id2item

In [ ]:
len(dm.item2id)

In [ ]:
len(dm.id2item)

In [ ]:
metadata_path = Path("../data/raw/2018/meta_Musical_Instruments.json.gz")
output_path = Path("../artifacts/item_embeddings/musical_instruments_content_embeddings.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
text_builder = AmazonItemTextBuilder(
    include_title=True,
    include_categories=True,
    include_features=True,
    include_description=True,
    include_details=True,
    include_price=False,
    max_chars=4096,
)

embedder = HashingTextEmbedder(
    dim=384,
    lowercase=True,
    use_bigrams=True,
    normalize=True,
)

builder = ContentEmbeddingBuilder(
    text_builder=text_builder,
    embedder=embedder,
    item_key="parent_asin",
)

result = builder.build_from_metadata(
    metadata_path=metadata_path,
    batch_size=512,
    allowed_item_ids=list(dm.item2id.keys()),
)

result.store.save(output_path)

print("Saved embeddings to:", output_path)
print("Number of items:", len(result.store.item_ids))
print("Embedding shape:", result.store.embeddings.shape)

In [ ]:
embeddings_path = Path("../artifacts/item_embeddings/musical_instruments_content_embeddings.pt")
output_dir = Path("../artifacts/tokenizers/rqvae_musical_instruments")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
embedding_store = ItemEmbeddingStore.load(embeddings_path)

item_ids = embedding_store.item_ids
embeddings = embedding_store.embeddings

print("Number of items:", len(item_ids))
print("Embedding shape:", embeddings.shape)

In [ ]:
rqvae_cfg = RQVAEConfig(
    input_dim=embeddings.shape[1],
    latent_dim=32,
    hidden_dims=(512, 256, 128),
    num_codebooks=3,
    codebook_size=256,
    commitment_weight=0.25,
    dropout=0.0,
    use_batch_norm=False,
)

train_cfg = RQVAETrainConfig(
    epochs=10,
    batch_size=1024,
    learning_rate=1e-3,
    weight_decay=0.0,
    val_ratio=0.05,
    seed=42,
    gradient_clip_norm=1.0,
    device="cpu",  # use "cpu" if needed
)

tokenizer_cfg = RQVAEItemTokenizerConfig(
    rqvae=rqvae_cfg,
    train=train_cfg,

    # Recommended for text embeddings.
    normalize="l2",

    # Important for TIGER-style flat token vocabulary:
    # code_l -> code_l + l * codebook_size
    level_offset_tokens=True,

    # Resolve duplicate semantic IDs by appending a suffix token.
    collision_policy="append",

    encode_batch_size=4096,
)


In [ ]:
tokenizer = RQVAEItemTokenizer(tokenizer_cfg)

tokenizer.fit(
    embeddings=embeddings,
    item_ids=item_ids,
)

In [ ]:
tokenizer.save(output_dir)

semantic_ids_path = output_dir / "semantic_ids.json"

print("Saved tokenizer to:", output_dir)
print("Saved semantic IDs to:", semantic_ids_path)

In [ ]:
semantic_id_store = tokenizer.semantic_id_store
semantic_id_mapping = semantic_id_store.to_plain_dict()
semantic_id_mapping

In [ ]:
len(semantic_id_mapping)

In [ ]:
dm.add_semantic_ids(semantic_ids=semantic_id_mapping)

In [ ]:
dm.semantic_ids

In [ ]:
len(dm.semantic_ids)

In [ ]:
for batch_idx, raw_batch in enumerate(dm.train_dataloader()):
    print(f"Batch {batch_idx}:")
    print(raw_batch)
    break